## Checking the fake decks vs real decks split worked



In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from models import common

decks = common.load_decks()

# fake data with a known answer (2.9 mana value, 17 lands) to test the code
fake_decks = common.fake_decks()

train, test = common.split_by_draft(decks)
print(len(decks), "decks:", len(train), "train,", len(test), "test")
decks.head()

106900 decks: 80264 train, 26636 test


,deck_id,draft_id,build_index,main_colors,wins,games,losses,num_cards,total_mana_value,num_lands,num_creatures,num_spells,avg_mana_value
0,deck_1,488a8410974649eb82add4b1a6f54a75,0,BR,3,6,3,40,69,16,16,24,2.875000
1,deck_2,41dafe993991494e9bbf2894af398d25,0,WR,6,9,3,40,55,16,15,24,2.291667
2,deck_3,f570c6a0e89a44ff83f9301b1fa8ef1b,0,WB,2,5,3,40,66,17,13,23,2.869565
3,deck_4,7c5e0736075241d7a0170c535f9a0cdf,0,WR,3,6,3,40,48,16,13,24,2.000000
4,deck_5,7d6e7ac9a9eb40808c54bd251b72837c,1,RG,5,8,3,40,66,17,18,23,2.869565


In [2]:
print(len(fake_decks), "decks:", len(train), "train,", len(test), "test")
fake_decks.head()

20000 decks: 80264 train, 26636 test


,draft_id,avg_mana_value,num_creatures,num_lands,games,wins,main_colors
0,0,2.950292,16,18,6,1,BG
1,1,2.847158,13,17,7,6,WB
2,2,3.156169,10,16,5,4,WB
3,3,2.941960,11,15,4,2,BG
4,4,2.685732,17,18,5,2,BG


In [3]:
# baseline = predict the average win rate for every deck
from models import baseline
base = baseline.fit(train)
print("baseline win rate:", base)
print("baseline error:", common.error_per_100(test, baseline.predict(base, test)))

baseline win rate: 0.5575263613930368
baseline error: 17.491505726431143


In [4]:
# main model, logistic regression in statsmodels
from models import statsmodels_logit
sml = statsmodels_logit.fit(train)
best = statsmodels_logit.best_values(sml, train)
for col, b in best.items():
    print(col, "| best:", b["best"], "| shape:", b["shape"])
print("error:", common.error_per_100(test, statsmodels_logit.predict(sml, test)))

avg_mana_value | best: 2.2 | shape: lower is better
num_creatures | best: 19.0 | shape: higher is better
num_lands | best: 16.0 | shape: peak
error: 17.418649818229092


In [5]:
# SGD at three learning rates
from models import sgd_logit
for alpha in [0.01, 0.1, 0.5]:
    f = sgd_logit.fit(train, alpha=alpha)
    print(alpha, common.error_per_100(test, sgd_logit.predict(f, test)))

0.01 17.42168872483598
0.1 17.44816431452064
0.5 17.557341171169842


In [6]:
# everything in one place
from models import report
results = report.final_results(train, test)

train decks: 80264  test decks: 26636

Intercept                -10.1352
C(main_colors)[T.BR]      -0.0274
C(main_colors)[T.RG]      -0.0840
C(main_colors)[T.UB]      -0.0891
C(main_colors)[T.UG]      -0.0465
C(main_colors)[T.UR]      -0.2050
C(main_colors)[T.WB]      -0.0257
C(main_colors)[T.WG]      -0.0141
C(main_colors)[T.WR]      -0.1161
C(main_colors)[T.WU]      -0.1542
avg_mana_value            -0.0333
I(avg_mana_value ** 2)    -0.0111
num_creatures             -0.0483
I(num_creatures ** 2)      0.0019
num_lands                  1.3408
I(num_lands ** 2)         -0.0412
dtype: float64

avg_mana_value: best 2.2 (lower is better)
     model   test
2.2  57.10  55.22
2.3  56.90  56.74
2.4  56.69  56.42
2.5  56.47  57.70
2.6  56.25  56.41
2.7  56.03  55.97
2.8  55.80  56.07
2.9  55.56  55.57
3.0  55.31  55.39
3.1  55.07  55.60
3.2  54.81  54.01
3.3  54.55  54.84
3.4  54.28  55.59
3.5  54.01  55.84

num_creatures: best 19 (higher is better)
      model   test
9.0   56.04  55.23
10.0  5